# VTU export and Paraview/pyvista visualization of a 3D solid

Demonstrates `write_bezier_patch_vtu` on a genuine **3D solid** patch, and visualizes
the result with pyvista: the undeformed vs. deformed (warped) mesh, and a clipped
cross-section to see the hollow interior.

This notebook deliberately reuses the octant-of-a-hollow-sphere NURBS geometry from
`20_nurbs_3d_sphere_octant.ipynb` — degree **(2, 2, 1)**, different per direction — because
rendering it is exactly what exposed two real, confirmed bugs in how this module wrote
VTU files for 3D solids: the written files were spec-compliant XML, VTK could open them
and read point data without complaint, but the cell *geometry* it reconstructed was
silently wrong (and, for an anisotropic degree like this one, decomposing it with
`warp_by_vector()`/`clip()` crashed outright). Both are fixed now — see the
*"VTU export and a VTK higher-order hexahedron bug"* section of the `future` module's
design docs for the full diagnosis. This notebook is the first one to actually exercise
the fix end to end on a 3D solid.

In [ ]:
import os

import numpy as np
import scipy.sparse.linalg as spla
import pyvista as pv

from yeti_iga.future.bspline import (
    BSpline, BSplineVolume, Patch, ControlPointManager,
    GlobalDOFManager, PatchDOFManager, IGABasis1D, Material, Elasticity3D,
    PatchIntegrator, ConstantTraction3D, SubdivisionRefiner,
)
from yeti_iga.future.postprocessing.vtu import write_bezier_patch_vtu
from yeti_iga.future.postprocessing.pv_plotting import (
    element_boundary_curves_3d, element_boundary_curves_3d_deformed,
)

## Geometry — octant of a hollow sphere (anisotropic degree), 4 elements

Same surface-of-revolution construction as notebook 20: a quarter-circle profile
(polar angle $u$, degree 2) swept through a quarter-circle rotation (azimuthal angle
$v$, degree 2) about the $z$-axis, with a plain affine (degree 1) radial thickness
direction $w$ — degree **(2, 2, 1)**, not all equal.

The raw construction below is a single Bezier element; `SubdivisionRefiner` then
bisects the two angular directions once each (4 angular elements, the radial direction
stays a single element since it never appears on the rendered boundary surface),
giving a genuine multi-element mesh to export and visualize below. Refining a `Patch`
that already has a `PatchDOFManager` attached updates it in place (true for a single,
non-shared patch like this one — see `02_patch_basics.ipynb`), so no DOF manager needs
rebuilding here.

In [ ]:
Rin, Rex = 1.0, 2.0
w0 = 1.0 / np.sqrt(2.0)
E, NU = 210000.0, 0.3

def profile(R):
    return [(0.0, R, 1.0), (R, R, w0), (R, 0.0, 1.0)]

rot = [(1.0, 0.0, 1.0), (1.0, 1.0, w0), (0.0, 1.0, 1.0)]

mgr = ControlPointManager(dim=3)
n_cp0 = 3 * 3 * 2
for radius in (Rin, Rex):
    prof = profile(radius)
    for iv in range(3):
        rx, ry, wv = rot[iv]
        for iu in range(3):
            x_p, z_p, wu = prof[iu]
            mgr.add_point([x_p * rx, x_p * ry, z_p], w=wu * wv)

su0 = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
sv0 = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
sw0 = BSpline(1, np.array([0., 0., 1., 1.]))
tensor0 = BSplineVolume(su0, sv0, sw0)

gdm = GlobalDOFManager([3] * n_cp0)
pdm = PatchDOFManager(3, list(range(n_cp0)), gdm)
patch = Patch(tensor0, mgr, list(range(n_cp0)), [3, 3, 2], pdm)

SubdivisionRefiner(direction=0, n_levels=1).refine(patch)
SubdivisionRefiner(direction=1, n_levels=1).refine(patch)

su, sv, sw = patch.tensor.components
n_cp = patch.n_cp

print(f'degrees: {[su.degree, sv.degree, sw.degree]}  (anisotropic: {len({su.degree, sv.degree, sw.degree}) > 1})')
print(f'n_cp: {n_cp}')

## A small structural problem

The $v{=}0$ face lies in the flat $y{=}0$ plane — clamp it entirely (homogeneous
Dirichlet, all 3 dofs). The $v{=}1$ face (flat $x{=}0$ plane) is pushed with a constant
traction in $z$. Both are flat, ordinary faces (not the curved inner/outer sphere
surfaces) — nothing exotic about the mechanics here, only the geometry's degree
pattern is unusual.

In [ ]:
law = Elasticity3D(Material(E=E, nu=NU))
basis_u = IGABasis1D.build(su, su.degree + 1)
basis_v = IGABasis1D.build(sv, sv.degree + 1)
basis_w = IGABasis1D.build(sw, sw.degree + 1)
integrator = PatchIntegrator(patch, basis_u, basis_v, basis_w, law)

K = integrator.integrate_stiffness().tocsr()
F = integrator.integrate_boundary_load_3d(1, 1, ConstantTraction3D(np.array([0.0, 0.0, 500.0])))

fixed_cp = np.array(sorted(patch.boundary_control_points(1, 0)))
fixed_dofs = np.sort(np.concatenate([3*fixed_cp, 3*fixed_cp+1, 3*fixed_cp+2]))
free_dofs = np.setdiff1d(np.arange(3 * n_cp), fixed_dofs)

u = np.zeros(3 * n_cp)
u[free_dofs] = spla.spsolve(K[free_dofs][:, free_dofs].tocsc(), F[free_dofs])
displacement = u.reshape(-1, 3)

print(f'Solved. max |u| = {np.max(np.abs(u)):.4e}')

## VTU export

`write_bezier_patch_vtu` now exports this degree-(2,2,1) patch directly — no
workaround, no cloning to an equal-degree copy. (Earlier versions of this function
needed one, to sidestep a VTK bug that turned out to share its root cause with the
point-ordering bug below; both are fixed at the source now.)

In [ ]:
os.makedirs('output', exist_ok=True)
write_bezier_patch_vtu(patch, 'output/sphere_octant.vtu', field=displacement, field_name='disp')
print('Exported -> output/sphere_octant.vtu')

## Visualization — undeformed vs. deformed

`warp_by_vector()` displaces every point by the `disp` field (exaggerated by `factor`
for visibility) — exactly the operation that used to crash on this geometry.

Both results are still VTK Bézier cells, not flat-faced linear ones — `add_mesh()` on
the raw grid would connect CONTROL points with straight edges (which, for a circular
arc, sit well outside the true curve) rather than the smooth surface they represent.
`extract_surface(nonlinear_subdivision=...)` tessellates the true curved Bézier surface
into enough flat triangles to look smooth — the same step the existing
`13_kirchhoff_love_shell.ipynb` notebook already uses for its curved shell renders.

A viewer's own "edges" toggle on that tessellated surface shows the triangulation, not
the patch's actual element boundaries. `element_boundary_curves_3d()` overlays the
*true* curved element grid instead, evaluated directly from the NURBS basis — the
3D-solid analogue of `element_boundary_curves()`, already used for 2D shells in
`13_kirchhoff_love_shell.ipynb`. With the patch now subdivided into 4 angular elements
(previous cell), this overlay actually traces a grid rather than just the outer
boundary.

In [ ]:
mesh = pv.read('output/sphere_octant.vtu')
warped = mesh.warp_by_vector('disp', factor=10.0)

mesh_surf = mesh.extract_surface(nonlinear_subdivision=5, algorithm=None)
warped_surf = warped.extract_surface(nonlinear_subdivision=5, algorithm=None)

edges_ref = element_boundary_curves_3d(patch, n_samples=20)
edges_deformed = element_boundary_curves_3d_deformed(patch, u, factor=10.0, n_samples=20)

pl = pv.Plotter(off_screen=True, window_size=[800, 650])
pl.add_mesh(mesh_surf, color='lightgray', opacity=0.3, label='undeformed')
pl.add_mesh(edges_ref, color='gray', line_width=2)
pl.add_mesh(warped_surf, scalars='disp', cmap='viridis', show_scalar_bar=True, label='deformed (x10)')
pl.add_mesh(edges_deformed, color='black', line_width=2)
pl.add_axes()
pl.view_isometric()
pl.camera.azimuth = 35
pl.camera.elevation = 15
pl.show(jupyter_backend='trame')

## Seeing the hollow interior

`mesh.tessellate()` turns the Bézier hexahedron into a true **volumetric** mesh of flat
tetrahedra/wedges (rather than just its boundary surface), so `clip()` on the result cuts
through the solid and exposes the interior — the ordinary pyvista way to do this. Both
operations decompose the cell's actual 3D geometry, exactly like the bugs above, so this
is also a direct check that the fix holds up away from the boundary surface, not only on
it.

In [ ]:
tess = mesh.tessellate(max_n_subdivide=4)
cross_section = tess.clip(normal='y', origin=(0.0, 1.e-3, 0.0))

pl = pv.Plotter(off_screen=True, window_size=[800, 650])
pl.add_mesh(cross_section, color='lightsteelblue', show_edges=False)
pl.add_axes()
pl.view_isometric()
pl.camera.azimuth = 35
pl.camera.elevation = 15
pl.show(jupyter_backend='trame')
print('The cut exposes the quarter-annulus cross-section -- wall thickness Rex-Rin is '
      'visible directly, from any angle by rotating the view above.')